In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.io.wavfile as wav
%matplotlib inline

## Άσκηση - Ευρωστία στην ανάλυση παθολογίας

Η άσκηση αυτή αποτελεί συνέχεια της άσκησης $\texttt{Ex-VoicePathology}$, σχετικά με παθολογία φωνής. 

Ίσως να σκεφτήκατε ότι αυτό που κάναμε σε εκείνη την άσκηση, δηλ. το να πάρουμε ένα τυχαίο κομμάτι απ΄ το σήμα φωνής μας και αφού το αναλύσουμε, να βγάλουμε απόφαση για κάτι τόσο σοβαρό όπως μια πιθανή παθολογία, είναι λίγο ριψοκίνδυνο και επιπόλαιο. 

Κάτι πιο ασφαλές θα ήταν το εξής:

$\left(\alpha'\right)$ Χωρίστε όλο το σήμα σε παράθυρα διάρκειας $50$ ms, με μια επικάλυψη γειτονικών παραθύρων της τάξης του $50\%$, δηλ. ‘‘προχωράτε’’ το παράθυρό σας πάνω στο σήμα της φωνής κάθε $25$ ms, ώστε τα παράθυρά σας να επικαλύπτονται κατά μισό παράθυρο.

Αυτό μπορείτε να το κάνετε με χρήση βρόχων επανάληψης όπως τους γνωρίζετε από τη `C` (for, while) - δε διαφέρουν πολύ. Μπορείτε να τις Google-άρετε για να δείτε πως συντάσσονται. Σκεφτείτε οτι απλά πρέπει να διατρέχετε ένα πίνακα-γραμμή (που είναι το σήμα σας) ανά κάποιο αριθμό στοιχείων.

$\left(\beta'\right)$ Υπολογίστε τον μετασχηματισμό Fourier για τις συχνότητες $2000−4000$ Hz με ανάλυση $Df = 1$ Hz, και βρείτε το φάσμα πλάτους του κάθε παραθύρου. Αποθηκεύστε το φάσμα πλάτους κάθε παραθύρου σε μια γραμμή ενός πίνακα $X$. Αυτό μπορεί να γίνει ως εξής (**ο παρακάτω κώδικας *δεν* είναι εκτελέσιμος, σας δίνεται ως βοήθεια για παρακάτω**):

```
for i in range(nseg):                              # Για κάθε παράθυρο στο σήμα...
                                                   # έστω segment η μεταβλητή που έχουμε αποθηκεύσει 
                                                   # το τρέχον παράθυρό μας
                                                   
    M = np.exp(-1j*2*np.pi*ff.T@tt)                # Πίνακας ανάλυσης Fourier 
    MF = (1/Fs)*segment@M.T                        # Μετασχηματισμός Fourier
    Fasma_platous = np.abs(MF)                     # Φάσμα πλάτους του
    Y[i, :] = Fasma_platous                        # Αποθήκευσή του σε μια λίστα
```

$\left(\gamma'\right)$ Υπολογίστε το "**μέσο φάσμα πλάτους**", δηλ. μια μέση τιμή όλων των φασμάτων πλάτους που έχετε βρει, έτσι ώστε στο τέλος να έχουμε μόνο ένα φάσμα πλάτους που αποτελει το μέσο όρο όλων, και να αποφασίσετε για την παθολογία βάσει αυτού. Χρήσιμη θα σας φανεί η εντολή $\texttt{mean}$ της $\texttt{NumPy}$.

Ακολουθώντας μια τέτοια διαδικασία έχουμε πιο εύρωστα, με τη στατιστική έννοια, συμπεράσματα.

Ας φορτώσουμε το σήμα μας.

In [ ]:
Fs, s = wav.read('./files/alpha.wav')        # Φορτώστε το αρχείο ήχου
s = s / np.max(np.abs(s))            # Κανονικοποίηση
L = len(s)                           # Διάρκεια του σήματος σε δείγματα

Ας ορίσουμε τις παραμέτρους της ανάλυσής μας.

In [ ]:
winlen_ms = 50e-3 # ms               # Έστω ότι το παράθυρό μας είναι 50 ms
winlen = np.round(winlen_ms*Fs) # INSERT CODE HERE          # Η διάρκειά του σε δείγματα
overlap = int(winlen/2)              # Επικάλυψη κατά 50% των διαδοχικών παραθύρων

Ας ξεκινήσουμε τη διαδικασία που περιγράψαμε παραπάνω.

In [ ]:
Df = 1                                               # Βήμα στη συχνότητα            
f = np.arange(2000, 4000, Df)                        # INSERT CODE HERE                               # Άξονας συχνότητας
ff = np.expand_dims(f, axis=0)                       # Επέκταση διάστασης όπως αναφέραμε στην 1η άσκηση Python
t = np.arange(start=0, stop=winlen_ms, step=1/Fs)    # Άξονας χρόνου για ένα παράθυρο διάρκειας 50ms
tt = np.expand_dims(t, axis=0)                       # Επέκταση διάστασης όπως αναφέραμε στην 1η άσκηση Python

N = int((L - winlen)/overlap)                  # Πλήθος frames που υπάρχουν στο σήμα μας
Y = np.zeros(shape=(N, len(f)))                # Δέσμευση μνήμης (όπως η calloc στη C)

for i in range(N):                             # Βρόχος επανάληψης που διατρέχει τα frames του σήματος
    start = i*overlap # INSERT CODE HERE                 # Έναρξη παραθύρου - χρησιμοποιήστε τη μεταβλητή overlap και το δείκτη i
    stop = i*overlap + winlen                  # Τέλος παραθύρου
    segment = s[int(start):int(stop)] # INSERT CODE HERE               # Αποκόπτουμε από το συνολικό σήμα s το κομμάτι που μας ενδιαφέρει - κάντε χρήση
                                               # των start και stop επάνω στο διάνυσμα s, θυμηθείτε πώς κάνουμε slicing ένα array στην Python
    M = np.exp(-1j * 2 * np.pi * ff.T @ tt) # INSERT CODE HERE                     # Πίνακας αναλυσης M(f,t) - όπως τον δείξαμε στην 1η άσκηση του εργαστηρίου
    
    MF = (1/Fs) * segment @ M.T                # Μετασχ. Fourier (όπως τον δείξαμε στην 1η άσκηση του εργαστηρίου)
    
    Y[i, :] = np.abs(MF)                       # Συλλέγουμε τα φάσματα πλάτους στον πίνακα Y, ένα σε κάθε θέση-γραμμή i του πίνακα

Έχοντας μαζέψει όλα τα φάσματα πλάτους στον πίνακα Υ, ένα φάσμα σε κάθε γραμμή του, μπορούμε να υπολογίσουμε τη μέση τιμή όλων των φασμάτων πλάτους!

In [ ]:
meanY = np.mean(Y, axis=0) # INSERT CODE HERE                     # Υπολογίζουμε τη μέση τιμή των φασμάτων πλάτους - χρησιμοποιήστε την np.mean(..., axis=0)

plt.figure(figsize=(12,4))                     # Απεικόνιση
plt.plot(f, meanY)                             # Γράφημα
plt.xlim([f[0], f[-1]])
plt.grid()
plt.xlabel("Συχνότητα (Hz)", fontsize=18)      # Ομορφαίνουμε
plt.ylabel("Πλάτος", fontsize=18)              # Ομορφαίνουμε
plt.title('Μέσο Φάσμα Πλάτους στο [2000, 4000]', fontsize=18)   # Ομορφαίνουμε
plt.show()

Αν τα κάνετε όλα σωστά, θα πάρετε - για το δικό μας σήμα - ένα σχήμα όπως παρακάτω:

![alt text](./files/FTpath2.png "Title")

Παρατηρούμε ότι τελικά 😌 δεν υπάρχει ιδιαίτερη συμμετρία γύρω από τα $3000$ Hz (για το δικό μας σήμα - για το δικό σας? 😁) Απαντήστε παρακάτω:

### Απάντηση:


---
---